In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel
import torch

HF_NAME = ""
MODEL_NAMME = ""
HF_TOKEN = ""
REPO_NAME = ""
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=f"{HF_NAME}/{MODEL_NAMME}",
    load_in_4bit=False,
    dtype=torch.float16, # colab 
)

model.save_pretrained_gguf("directory", tokenizer, quantization_method="q8_0") # Use quan 8 for smaller size, q4_0 for better performance but larger size

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

api.create_repo(
    repo_id=f"{HF_NAME}/{MODEL_NAMME}-gguf",
    repo_type="model",
    private=False,  # True nếu muốn private
    token=f"{HF_TOKEN}",
)

api.upload_file(
    path_or_fileobj=f"directory_gguf/{MODEL_NAMME}.Q8_0.gguf",
    path_in_repo=f"{MODEL_NAMME}.Q8_0.gguf",
    repo_id=f"{HF_NAME}/{MODEL_NAMME}-gguf",
    repo_type="model",
    token=f"{HF_TOKEN}",
)